In [3]:

import re

def is_variable(t):
    return isinstance(t, str) and t[0].islower()


def is_constant(t):
    return isinstance(t, str) and t[0].isupper()


def to_string(t):
    if isinstance(t, str):
        return t
    return t[0] + '(' + ', '.join(to_string(a) for a in t[1:]) + ')'


def show(subst):
    if subst is None:
        return 'FAILURE'
    return '{' + ', '.join(f'{to_string(v)}/{k}' for k, v in subst.items()) + '}'


def parse(text):
    tokens = re.findall(r"[A-Za-z0-9_']+|[(),]", text)
    term, _ = parse_term(tokens, 0)
    return term


def parse_term(tokens, pos):
    name = tokens[pos]
    pos += 1
    if pos < len(tokens) and tokens[pos] == '(':
        pos += 1
        args = []
        while True:
            arg, pos = parse_term(tokens, pos)
            args.append(arg)
            if tokens[pos] == ',':
                pos += 1
            else:
                pos += 1
                break
        return (name,) + tuple(args), pos
    return name, pos


def occurs(var, term):
    """True if variable var appears anywhere inside term."""
    if var == term:
        return True
    if isinstance(term, tuple):
        return any(occurs(var, arg) for arg in term[1:])
    return False


def apply_subst(subst, term):
    """Return term with every variable in subst replaced by its value."""
    if isinstance(term, str):
        return subst.get(term, term)
    return (term[0],) + tuple(apply_subst(subst, arg) for arg in term[1:])


def append_subst(s, subst):
    """Merge new substitution s into the existing set subst."""
    merged = {v: apply_subst(s, t) for v, t in subst.items()}
    merged.update(s)
    return merged


def unify(x, y):

    if x == y:
        return {}


    if is_variable(x):
        if occurs(x, y):
            return None
        return {x: y}


    if is_variable(y):
        if occurs(y, x):
            return None
        return {y: x}


    if is_constant(x) or is_constant(y):
        return None


    if x[0] != y[0]:
        return None


    if len(x) != len(y):
        return None


    subst = {}
    for i in range(1, len(x)):
        x_i = apply_subst(subst, x[i])
        y_i = apply_subst(subst, y[i])
        s = unify(x_i, y_i)
        if s is None:
            return None
        if s:
            subst = append_subst(s, subst)
    return subst



def run(e1, e2):
    result = unify(parse(e1), parse(e2))
    print(f'Unify( {e1} , {e2} )')
    if result is None:
        print('   Result : FAILURE (cannot be unified)\n')
    else:
        print('   Substitution :', show(result))
        print('   Unified expression :', to_string(apply_subst(result, parse(e1))), '\n')


if __name__ == '__main__':
    run('Eats(x, Apple)', 'Eats(Riya, y)')              # success
    run('Knows(John, x)', 'Knows(x, Elisabeth)')        # fail: x cannot be both
    run('P(f(A), g(y))', 'P(x, x)')                     # fail: f and g differ
    run('P(A, h(y))', 'P(A, f(z))')                     # fail: h and f differ
    run('Knows(John, x)', 'Knows(y, Mother(y))')        # nested function





Unify( Eats(x, Apple) , Eats(Riya, y) )
   Substitution : {Riya/x, Apple/y}
   Unified expression : Eats(Riya, Apple) 

Unify( Knows(John, x) , Knows(x, Elisabeth) )
   Result : FAILURE (cannot be unified)

Unify( P(f(A), g(y)) , P(x, x) )
   Result : FAILURE (cannot be unified)

Unify( P(A, h(y)) , P(A, f(z)) )
   Result : FAILURE (cannot be unified)

Unify( Knows(John, x) , Knows(y, Mother(y)) )
   Substitution : {John/y, Mother(John)/x}
   Unified expression : Knows(John, Mother(John)) 

